# 03b – Preprocessing v2 (erweiterte Features)

**Vergleich zu v1:** [docs/10_PREPROCESSING_V2.md](../docs/10_PREPROCESSING_V2.md) · Original: `03_preprocessing.ipynb`

| Output v2 | v1 (alt) |
|-----------|----------|
| `train_labeled_v2.parquet` | `train_labeled.parquet` |
| `test_features_v2.parquet` | `test_features.parquet` |

Neu: Score-Lags, `score_persist7` als Feature, Regional-Score-Stats, 91-Tage-Test-Aggregate.

**Parallel:** mehrere Regionen gleichzeitig (`DM_WORKERS`, Standard ≈ CPU−1). Colab-Tipp: CSV nach `/content/` kopieren (Zelle 2).

→ danach **`04b_modeling_v2.ipynb`**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for _p in (Path.cwd(), Path.cwd().parent):
    _r = _p.resolve()
    if (_r / "scripts" / "notebook_init.py").exists() and str(_r) not in sys.path:
        sys.path.insert(0, str(_r))
        break

from scripts.notebook_init import setup
from scripts.project_env import load_script

env = setup()
PROJECT_ROOT = env["project_root"]
TRAIN_PATH = env["train_path"]
TEST_PATH = env["test_path"]
PROCESSED_DIR = env["processed_dir"]
MODE = env["mode"]

_fv2 = load_script("features_v2", PROJECT_ROOT / "scripts" / "features_v2.py")
feature_columns_v2 = _fv2.feature_columns_v2
build_features_v2 = _fv2.build_features_v2

print("Setup OK | MODE:", MODE, "| Features v2:", len(feature_columns_v2()))

In [ ]:
import os
import shutil

from scripts.parallel_util import default_workers

path_train_v2 = PROCESSED_DIR / "train_labeled_v2.parquet"
path_test_v2 = PROCESSED_DIR / "test_features_v2.parquet"
N_WORKERS = default_workers()

# Colab: Drive-I/O vermeiden (oft schneller als Paid-CPU)
train_path = TRAIN_PATH
test_path = TEST_PATH
if env["is_colab"]:
    local_train = Path("/content/train.csv")
    local_test = Path("/content/test.csv")
    if not local_train.exists():
        print("Kopiere train.csv → /content/ …")
        shutil.copy(TRAIN_PATH, local_train)
    if not local_test.exists():
        print("Kopiere test.csv → /content/ …")
        shutil.copy(TEST_PATH, local_test)
    train_path, test_path = local_train, local_test

_ps2 = load_script("preprocess_v2", PROJECT_ROOT / "scripts" / "preprocess_streaming_v2.py")

if MODE == "full":
    print(f"Streaming v2 parallel ({N_WORKERS} workers) …")
    stats = _ps2.preprocess_by_region_v2(
        train_path, test_path, path_train_v2, path_test_v2,
        chunk_size=500_000, n_workers=N_WORKERS,
    )
    print("Fertig:", stats)
else:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    train_labeled, test_feat = _ps2.preprocess_panel_v2(train, test, n_workers=N_WORKERS)
    train_labeled.to_parquet(path_train_v2, index=False)
    test_feat.to_parquet(path_test_v2, index=False)
    print(f"Gespeichert: {len(train_labeled):,} train | {len(test_feat):,} test")

In [ ]:
import pyarrow.parquet as pq

n_train = pq.read_metadata(path_train_v2).num_rows
n_test = pq.read_metadata(path_test_v2).num_rows
print(f"Parquet v2: train {n_train:,} | test {n_test:,}")

head = pd.read_parquet(path_train_v2)
v2_only = [c for c in head.columns if c.startswith(("score_lag", "region_score", "test91_"))]
print(f"Neue Spalten (Beispiel): {v2_only[:8]} … ({len(v2_only)} v2-spezifisch sichtbar)")
head.head(3)